# fase_3 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Migrasi Rekrutmen & Pelamar dengan Mapping Kolom Spesifik

In [ ]:
import sys
import os
import mysql.connector
import pandas as pd
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [ ]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

## 2. Ambil Data dari DB Lama

In [ ]:
hanif_tables_map = [
    ('pengajuan', 'pengajuan_karyawan'),
    ('histori_pengajuan', 'histori_pengajuan'),
    ('pelamar', 'pelamar'),
    ('pekerjaan', 'pelamar_kerja'),
    ('pendidikan', 'pelamar_sekolah'),
    ('kursus', 'pelamar_kursus'),
    ('pelamar_note', 'progres_pelamar'),
    ('pelamar_users', 'rekrutmen_pelamar')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    cursor_old.execute(f"SELECT * FROM `{old_t}`")
    raw_data[old_t] = cursor_old.fetchall()
    print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")

## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [ ]:
transformed_dfs = {}

# 1. pengajuan -> pengajuan_karyawan
if 'pengajuan' in raw_data:
    df = pd.DataFrame(raw_data['pengajuan'])
    mapping = {
        'idpengajuan': 'id_pengajuan', 'keterangan': 'id_user', 'syarat': 'posisi',
        'pertanyaan': 'jumlah', 'alur': 'syarat', 'test': 'pertanyaan',
        'status': 'alur_seleksi', 'jumlah': 'daftar_tes', 'idusers': 'status',
        'created_at': 'created_at'
    }
    transformed_dfs['pengajuan_karyawan'] = df.rename(columns=mapping)[list(mapping.values())]

# 2. histori_pengajuan -> histori_pengajuan
if 'histori_pengajuan' in raw_data:
    df = pd.DataFrame(raw_data['histori_pengajuan'])
    mapping = {
        'idhistori': 'id_verifikasi', 'status': 'id_pengajuan',
        'catatan': 'status_verifikasi_pengajuan', 'idpengajuan': 'catatan',
        'created_at': 'created_at'
    }
    transformed_dfs['histori_pengajuan'] = df.rename(columns=mapping)[list(mapping.values())]

# 3. pelamar -> pelamar
if 'pelamar' in raw_data:
    df = pd.DataFrame(raw_data['pelamar'])
    mapping = {
        'idpelamar': 'id_pelamar', 'email': 'id_pengajuan', 'nama': 'email_pelamar',
        'panggilan': 'nama_lengkap', 'jk': 'nama_panggilan', 'ttl': 'jenis_kelamin',
        'domisili': 'tempat_lahir', 'alamat': 'tanggal_lahir', 'wa': 'alamat_ktp',
        'sosmed': 'alamat_domisili', 'linkedin': 'nomor_wa', 'laptop': 'akun_linkedin',
        'internet': 'akun_instagram', 'kegiatan': 'akun_facebook', 'rencana': 'sosmed_lain',
        'mobilitas': 'spesifikasi_laptop', 'info': 'internet', 'wfo': 'kegiatan_sekarang',
        'bergabung': 'rencana_karir', 'jenis': 'mobilitas', 'created_at': 'sumber_info',
        'status': 'siap_wfo', 'ig': 'tanggal_bergabung', 'fb': 'kategori_pelamar',
        'idpengajuan': 'riwayat_kerja', 'work': 'riwayat_pendidikan',
        'ppdk': 'pengalaman_bidang', 'pengalaman': 'wawasan', 'wawasan': 'riwayat_kesehatan',
        'sehat': 'status_pernikahan', 'statusnikah': 'kemampuan_ajar',
        'ajar': 'penguasaan_aplikasi', 'app': 'aplikasi_lainnya', 'gunalaptop': 'penggunaan_laptop',
        'toefl': 'skor_toefl', 'apps': 'ekspektasi_gaji', 'gaji': 'tautan_berkas',
        'link': 'alasan_resign', 'resign': 'skor_iq', 'hasiliq': 'foto_iq',
        'piciq': 'foto_minat', 'picminat': 'foto_kepribadian', 'picpribadi': 'created_at'
    }
    transformed_dfs['pelamar'] = df.rename(columns=mapping)[list(mapping.values())]

# 4. pekerjaan -> pelamar_kerja
if 'pekerjaan' in raw_data:
    df = pd.DataFrame(raw_data['pekerjaan'])
    mapping = {
        'idpekerjaan': 'id_pelamar_kerja', 'namaperusahaan': 'id_pelamar',
        'periode': 'nama_perusahaan', 'jabatan': 'periode', 'jobdesk': 'jabatan',
        'idusers': 'deskripsi_kerja'
    }
    transformed_dfs['pelamar_kerja'] = df.rename(columns=mapping)[list(mapping.values())]

# 5. pendidikan -> pelamar_sekolah
if 'pendidikan' in raw_data:
    df = pd.DataFrame(raw_data['pendidikan'])
    mapping = {
        'idpendidikan': 'id_pelamar_sekolah', 'sekolah': 'id_pelamar',
        'jenjang': 'nama_sekolah', 'prodi': 'jenjang', 'tahun': 'prodi',
        'ipk': 'tahun_lulus', 'idusers': 'ipk', 'organisasi': 'organisasi'
    }
    transformed_dfs['pelamar_sekolah'] = df.rename(columns=mapping)[list(mapping.values())]

# 6. kursus -> pelamar_kursus
if 'kursus' in raw_data:
    df = pd.DataFrame(raw_data['kursus'])
    mapping = {
        'idkursus': 'id_pelamar_kursus', 'nama': 'id_pelamar',
        'tanggal': 'nama_kursus', 'deskripsi': 'tanggal', 'lokasi': 'deskripsi',
        'nosertifikat': 'lokasi', 'idusers': 'nomor_sertifikat'
    }
    transformed_dfs['pelamar_kursus'] = df.rename(columns=mapping)[list(mapping.values())]

# 7. pelamar_note -> progres_pelamar
if 'pelamar_note' in raw_data:
    df = pd.DataFrame(raw_data['pelamar_note'])
    mapping = {
        'idnote': 'id_progres_pelamar', 'idpelamar': 'id_pelamar',
        'status': 'id_user', 'note': 'status_progres_pelamar',
        'idusers': 'catatan', 'created_at': 'tautan_file',
        'link': 'pertanyaan', 'pertanyaan': 'created_at'
    }
    transformed_dfs['progres_pelamar'] = df.rename(columns=mapping)[list(mapping.values())]

# 8. pelamar_users -> rekrutmen_pelamar
if 'pelamar_users' in raw_data:
    df = pd.DataFrame(raw_data['pelamar_users'])
    mapping = {
        'idassign': 'id_rekrutmen', 'idpelamar': 'id_pelamar', 'idusers': 'id_user'
    }
    transformed_dfs['rekrutmen_pelamar'] = df.rename(columns=mapping)[list(mapping.values())]

print(f"✓ Transformasi {len(transformed_dfs)} tabel Fase 3 selesai.")

## 4. Export ke Pickle

In [ ]:
file_name = 'fase_3_hanif.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(transformed_dfs, f)

total_records = sum(len(df) for df in transformed_dfs.values())
migration_result = {
    'fase': 'fase_3',
    'script': 'script_hanif',
    'fase_num': 3,
    'status': 'ready_for_insert',
    'records_transformed': total_records,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()